# Nestlé WISER DOM — final experiment laboratory

Clean final-report notebook. Run top-to-bottom. Edit only the configuration block in the code cell.

**Terminology:** QAOA is the quantum proposal algorithm. `ibm_marrakesh` is a QPU execution backend, not a solver. SciPy/HiGHS, native HiGHS, SCIP, and Gurobi are MILP backends for the same classical formulation.


In [ ]:
# =============================================================================
# IMPORTS + FINAL HIGH-BUDGET CONFIGURATION
# =============================================================================
import importlib.util, subprocess, sys, os
from pathlib import Path
from dataclasses import asdict, replace
import pandas as pd
from IPython.display import Image, Markdown, display

def root(start):
    for p in [Path(start).resolve(), *Path(start).resolve().parents]:
        if (p/"pyproject.toml").is_file() and (p/"src/domopt").is_dir(): return p
    raise RuntimeError("Open this notebook inside wiser-dom-optimization")

PROJECT_ROOT=root(Path.cwd())
if str(PROJECT_ROOT/"src") not in sys.path: sys.path.insert(0,str(PROJECT_ROOT/"src"))
if any(importlib.util.find_spec(x) is None for x in ("numpy","pandas","scipy","yaml","matplotlib","nbformat")):
    subprocess.run([sys.executable,"-m","pip","install","-e",f"{PROJECT_ROOT}[notebook]"],check=True)
for n in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS"): os.environ[n]="16"

from domopt.checkpoints import StaleCheckpointError,challenge_results_root,checkpoint_identity,checkpoint_run_directory,load_checkpoint,write_checkpoint
from domopt.classical import ClassicalSolverError,available_milp_backends,solve_classical
from domopt.experiments import experiment_profile,ibm_hardware_study_logical_qubits,make_ibm_hardware_study_problem,rank_ibm_hardware_strategies,run_challenge_experiments,run_ibm_hardware_study,write_experiment_results
from domopt.hardware import benchmark_qubo_batch_scoring,discover_ibm_backends,hardware_capabilities
from domopt.metrics import compute_metrics
from domopt.poc import POC_REFERENCE_FILENAMES,PocConfig,audit_poc_bundle,audit_poc_outputs,load_poc_problem,prune_pareto_candidates,select_shortage_subset
from domopt.visualization import plot_challenge_results,plot_hardware_benchmark,plot_ibm_backend_snapshot,plot_ibm_hardware_study

BUNDLE_DIR=(PROJECT_ROOT/"data/raw/nestle_challenge").resolve()
PROFILE="full"; FORCE_RERUN=False
POC_SETTINGS=PocConfig(protection_days=5,min_divert_improvement_fraction=.05,min_divert_improvement_cases=100,candidate_dc_scope="network_intersection",pareto_prune=False)

PROFILE_OVERRIDES={
"sizes":(8,20,50,100,150,200,250,300,372),
"synthetic_sizes":(50,100,250,500,1_000,2_000,5_000,10_000,20_000,50_000,100_000),
"scaling_repetitions":3,"base_orders":100,"exact_max_orders":100,"hybrid_max_orders":20_000,"lns_max_orders":372,
"inventory_shocks":(0,.10,.25,.40,.50,.55,.60,.65,.70),
"seeds":(3,11,29,47,71,101,131,173),
"noise_levels":(0,.01,.03,.05,.075,.10,.15,.20),
"readout_noise_levels":(0,.005,.01,.02,.03,.05,.075,.10),
"penalty_scales":(.25,.50,1.,2.,4.,8.),"candidate_counts":(1,2,3,4,5,6),
"qubo_one_hot_multipliers":(1.05,1.15,1.25,1.50,2.),"qubo_pair_multipliers":(0,.5,1.,1.5,2.),
}
HYBRID_OVERRIDES={"iterations":10,"neighborhood_orders":8,"max_qubo_variables":40,"max_candidates_per_order":5,"num_reads":256,"sweeps":300,"top_k_recourse":8,"recourse_time_limit_seconds":20.,"one_hot_penalty_multiplier":1.25,"pair_penalty_multiplier":1.,"batch_strategy":"conflict","polish_initial_incumbent":True,"qaoa_layers":1,"qaoa_restarts":16,"qaoa_mixer_topology":"path","milp_backend":"scipy-highs","thread_count":None,"seed":11}
EXACT_LNS_OVERRIDES={"iterations":16,"minimum_neighborhood_groups":4,"initial_neighborhood_groups":12,"maximum_neighborhood_groups":32,"maximum_neighborhood_orders":120,"maximum_local_fulfillment_variables":25_000,"local_time_limit_seconds":30.,"mip_relative_gap":.001,"diversification_interval":3,"adaptive":True,"polish_initial_incumbent":True,"milp_backend":"scipy-highs","thread_count":None,"seed":11}
EXPERIMENTS=("solver_comparison","size_scaling","synthetic_scaling","candidate_dc_scope_sensitivity","penalty_weight_sensitivity","qubo_penalty_sensitivity","candidate_count_sensitivity","inventory_shock","qubo_coefficient_noise","qaoa_readout_noise","pareto_pruning_ablation","batch_strategy_ablation","sampler_ablation","synthetic_coordination_control")
ENABLE_GPU_BENCHMARK=True
ENABLE_IBM_HARDWARE=False  # change to True only when intentionally submitting remote QPU jobs
IBM_HARDWARE_PROFILE="presentation"; IBM_SHOTS=8_192; IBM_BACKEND_NAME="ibm_marrakesh"
OUTPUT_ROOT=challenge_results_root(PROJECT_ROOT,producer="notebook")

# =============================================================================
# MODEL + VERIFIED CHECKPOINT HELPERS
# =============================================================================
audit=audit_poc_bundle(BUNDLE_DIR); assert audit["readable"].all(); display(audit[["role","filename","rows","columns","readable"]])
problem=load_poc_problem(BUNDLE_DIR,config=POC_SETTINGS,strict_bundle_audit=False)
pruned=prune_pareto_candidates(problem)
display(pd.DataFrame([{"variant":"unpruned","orders":len(problem.orders),"assignment_groups":problem.orders["assignment_group"].nunique(),"order_lines":len(problem.order_lines),"candidate_rows":len(problem.candidates)},{"variant":"pareto_pruned","orders":len(pruned.orders),"assignment_groups":pruned.orders["assignment_group"].nunique(),"order_lines":len(pruned.order_lines),"candidate_rows":len(pruned.candidates)}]))
if all((BUNDLE_DIR/n).is_file() for n in POC_REFERENCE_FILENAMES.values()): display(pd.Series(audit_poc_outputs(BUNDLE_DIR,problem),name="reference audit"))

BASE=experiment_profile(PROFILE)
SETTINGS=replace(BASE,**PROFILE_OVERRIDES,hybrid=replace(BASE.hybrid,**HYBRID_OVERRIDES),exact_lns=replace(BASE.exact_lns,**EXACT_LNS_OVERRIDES))
CONFIG={"experiment_profile":asdict(SETTINGS),"poc":asdict(POC_SETTINGS)}
OUT=checkpoint_run_directory(OUTPUT_ROOT,checkpoint_identity(problem,profile=PROFILE,experiment="challenge_suite",configuration=CONFIG))
TABLE=OUT/"tables"; FIG=OUT/"figures"; TABLE.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)

def feasible(df):
    x=df["feasible"]; return x if x.dtype==bool else x.astype(str).str.lower().isin({"true","1"})
def expected_fixed_infeasible(df):
    et=df["error_type"] if "error_type" in df else pd.Series("",index=df.index,dtype=str)
    em=df["error_message"] if "error_message" in df else pd.Series("",index=df.index,dtype=str)
    return df["experiment"].eq("inventory_shock") & df["level"].str.contains("fixed_routing_recourse",na=False) & et.eq("ClassicalSolverError") & em.str.contains(r"status=2|infeasible",case=False,na=False,regex=True)
def verify(df):
    bad=~feasible(df)
    if not bad.any(): return
    allowed=expected_fixed_infeasible(df); unexpected=df.loc[bad & ~allowed]
    if not unexpected.empty:
        cols=[c for c in ("experiment","level","validation_categories","validation_violation_count","error_type","error_message") if c in unexpected]
        raise RuntimeError(f"Unexpected infeasible rows: {unexpected[cols].to_dict('records')}")
    print("Expected fixed-policy infeasibility:",df.loc[allowed,[c for c in ("level","error_type","error_message") if c in df]].to_dict("records"))
def run_or_load(name):
    ident=checkpoint_identity(problem,profile=PROFILE,experiment=name,configuration=CONFIG); path=TABLE/f"{name}.csv"; df=None
    if not FORCE_RERUN:
        try: df=load_checkpoint(path,ident); print("loaded",path.relative_to(PROJECT_ROOT))
        except StaleCheckpointError as e: print(f"stale/missing {name}: {e}")
    if df is None:
        df=run_challenge_experiments(problem,profile=SETTINGS,experiments=[name]); write_experiment_results(df,path); write_checkpoint(df,path,ident); print("wrote",path.relative_to(PROJECT_ROOT))
    verify(df); return df

# =============================================================================
# 14-FAMILY MAIN STUDY + COMPLETE AGGREGATION
# =============================================================================
frames={}
for name in EXPERIMENTS:
    print(f"\n=== {name} ==="); df=run_or_load(name); frames[name]=df; display(df.head(200))

verified=[]
for name in EXPERIMENTS:
    ident=checkpoint_identity(problem,profile=PROFILE,experiment=name,configuration=CONFIG)
    df=load_checkpoint(TABLE/f"{name}.csv",ident); verify(df); verified.append(df.dropna(axis=1,how="all"))
results=pd.concat(verified,ignore_index=True,sort=False)
agg_id=checkpoint_identity(problem,profile=PROFILE,experiment="aggregate_results",configuration=CONFIG)
agg=write_experiment_results(results,OUT/"aggregate_results.csv"); write_checkpoint(results,agg,agg_id)
figs=plot_challenge_results(results,FIG)
print(f"aggregate rows: {len(results)}")
for name,path in figs.items(): display(Markdown(f"### {name.replace('_',' ').title()}")); display(Image(filename=str(path)))

# =============================================================================
# CORRECT MILP BACKEND COMPARISON
# =============================================================================
bp=select_shortage_subset(problem,SETTINGS.base_orders); avail=available_milp_backends(); rows=[]
for b in ("scipy-highs","highspy","scip","gurobi"):
    if not avail.get(b,False): rows.append({"milp_backend":b,"status":"not installed","feasible":None}); continue
    try: s=solve_classical(bp,backend=b,time_limit_seconds=180.,mip_relative_gap=.001,seed=SETTINGS.exact_lns.seed,thread_count=16)
    except ClassicalSolverError as e: rows.append({"milp_backend":b,"status":f"unavailable/error: {e}","feasible":None}); continue
    m=compute_metrics(bp,s); rows.append({"milp_backend":b,"status":"completed","feasible":m["feasible"],"objective_value":m["objective_value"],"runtime_seconds":m["runtime_seconds"],"optimality_gap":m["optimality_gap"],"best_bound":s.metadata.get("best_bound"),"mip_node_count":s.metadata.get("mip_node_count"),"solver":s.metadata.get("solver")})
backend=pd.DataFrame(rows); done=backend.loc[backend["status"].eq("completed")].copy()
if not done.empty:
    assert done["feasible"].all(); best=done["objective_value"].max()
    done["objective_gap_to_best"]=best-done["objective_value"]; done["objective_gap_to_best_pct"]=done["objective_gap_to_best"]/max(1.,abs(best))*100
    backend=backend.merge(done[["milp_backend","objective_gap_to_best","objective_gap_to_best_pct"]],on="milp_backend",how="left")
    proved=done["optimality_gap"].fillna(float("inf")).abs().le(1e-9)
    if len(done)>=2 and proved.all():
        spread=done["objective_value"].max()-done["objective_value"].min()
        assert abs(spread)<=1e-6*max(1.,done["objective_value"].abs().max())
backend.to_csv(TABLE/"milp_backend_comparison.csv",index=False); display(backend)

# =============================================================================
# GPU QUBO-SCORING CROSSOVER
# =============================================================================
display(pd.Series(hardware_capabilities(),name="hardware capability"))
if ENABLE_GPU_BENCHMARK:
    h=benchmark_qubo_batch_scoring(variable_counts=(16,32,40,64,96,128,256,512,1024),sample_counts=(256,4096,16384,65536,131072),repeats=7,seed=7,include_gpu=True)
    h.to_csv(OUT/"hardware_qubo_scoring.csv",index=False); display(h)
    display(Image(filename=str(plot_hardware_benchmark(h,FIG/"hardware_qubo_scoring.png"))))

# =============================================================================
# OPTIONAL IBM QAOA HARDWARE STUDY
# =============================================================================
if ENABLE_IBM_HARDWARE:
    nq=ibm_hardware_study_logical_qubits(); snap=discover_ibm_backends(min_num_qubits=nq); selected=IBM_BACKEND_NAME
    if selected not in set(snap["backend"].astype(str)): raise RuntimeError(f"QPU backend {selected!r} not eligible")
    snap["selected_for_study"]=snap["backend"].astype(str).eq(selected); display(snap)
    qp=make_ibm_hardware_study_problem()
    ident=checkpoint_identity(qp,profile=f"ibm-{IBM_HARDWARE_PROFILE}",experiment="ibm_hardware_stress",configuration={"qpu_backend":selected,"shots":IBM_SHOTS,"hardware_profile":IBM_HARDWARE_PROFILE,"logical_qubits":nq,"matrix":"QAOA p=1,2 x baseline,DD,DD+measurement-twirling","data_scope":"generated synthetic control"})
    qout=checkpoint_run_directory(OUTPUT_ROOT,ident); t=qout/"tables"; f=qout/"figures"; t.mkdir(parents=True,exist_ok=True); f.mkdir(parents=True,exist_ok=True)
    snap.to_csv(qout/"ibm_backend_snapshot.csv",index=False); display(Image(filename=str(plot_ibm_backend_snapshot(snap,f/"ibm_backend_queue.png"))))
    target=t/"ibm_hardware_stress.csv"; existing=None
    if not FORCE_RERUN:
        try: existing=load_checkpoint(target,ident); print(f"resuming from {len(existing)} verified rows")
        except StaleCheckpointError as e: print("no reusable IBM checkpoint:",e)
    q=run_ibm_hardware_study(allow_remote=True,backend_name=selected,shots=IBM_SHOTS,profile=IBM_HARDWARE_PROFILE,progress_callback=lambda x:write_checkpoint(x,target,ident),existing_results=existing)
    write_checkpoint(q,target,ident); ranking=rank_ibm_hardware_strategies(q); write_experiment_results(ranking,t/"ibm_strategy_ranking.csv")
    display(q); display(ranking); display(Image(filename=str(plot_ibm_hardware_study(q,f/"ibm_hardware_stress.png"))))
else:
    print("IBM hardware study skipped. Set ENABLE_IBM_HARDWARE=True to submit the documented 18-job QAOA matrix.")


## Interpretation guardrails

- Production path: greedy + exact recourse, then exact LNS when additional search is justified.
- Hybrid is a bounded assignment-proposal layer; exact recourse and validation retain feasibility authority.
- QAOA is the quantum algorithm. IBM device names are QPU execution backends, not solvers.
- Severe-shock frozen-routing infeasibility is a policy-breakdown result.
- Synthetic controls support algorithmic/scaling claims, not realized business-impact claims.
- GPU results concern QUBO scoring only.
- Do not claim quantum advantage; compare raw QAOA proposal quality against strong classical controls.
